In [3]:
# EQuIS_Well_Data_Relationships_v6.py
# Generates a PNG diagram showing EQuIS table relationships:
# DT_LOCATION (sys_loc_code parent), DT_WELL, DT_COORDINATE, DT_MEASURED_DATUM, DT_WELL_SEGMENT,
# plus the "Derived Calculations" panel. DT_LOCATION and DT_WELL are 3x wider and 2x taller.

from PIL import Image, ImageDraw, ImageFont

# ----------------------------
# Configuration (easy tweak zone)
# ----------------------------
CANVAS_W, CANVAS_H = 2000, 1500
TITLE = "EQuIS Data Relationships — Location, Coordinates, TOC, Construction, and Water Levels"

# Box styling
BOX_FILL = "#F6FAFF"
BOX_OUTLINE = "#2F5597"
TITLE_BAR_FILL = "#E7F0FF"
TITLE_TEXT = "#153E75"

# Calculations box styling
CALC_FILL = "#E5F7EA"
CALC_OUTLINE = "#1D6F42"

# Arrow & footer styling
ARROW_COLOR = "#666666"
FOOTER_TEXT = "Note: DT_LOCATION is the parent for all well data. Raw depths and survey values are preserved; elevations are always derived."
FOOTER_FILL = "#444444"

# Output filename
OUTFILE = "EQuIS_Well_Data_Relationships_v6.png"

# ----------------------------
# Canvas & Fonts
# ----------------------------
img = Image.new("RGB", (CANVAS_W, CANVAS_H), "white")
draw = ImageDraw.Draw(img)

def load_fonts():
    """Try to load DejaVu fonts, fall back to default if not available."""
    try:
        title_f = ImageFont.truetype("DejaVuSans-Bold.ttf", 48)
        h_f = ImageFont.truetype("DejaVuSans-Bold.ttf", 34)
        b_f = ImageFont.truetype("DejaVuSans.ttf", 26)
        small_f = ImageFont.truetype("DejaVuSans.ttf", 22)
    except Exception:
        title_f = ImageFont.load_default()
        h_f = ImageFont.load_default()
        b_f = ImageFont.load_default()
        small_f = ImageFont.load_default()
    return title_f, h_f, b_f, small_f

TITLE_FONT, HEADER_FONT, BODY_FONT, SMALL_FONT = load_fonts()

# ----------------------------
# Utilities
# ----------------------------
def multiline(text, x, y, font, fill, max_w=None, bullet=False, spacing=6):
    """
    Draw wrapped text (optionally bulleted) within max_w.
    Returns the new y position after drawing the lines.
    """
    lines = []
    if max_w:
        words = text.split()
        line = ""
        # Reserve a little space for the bullet glyph width
        bullet_pad = 30 if bullet else 0
        for w in words:
            test = (line + (" " if line else "") + w)
            if draw.textlength(test, font=font) <= (max_w - bullet_pad):
                line = test
            else:
                lines.append(line)
                line = w
        if line:
            lines.append(line)
    else:
        lines = text.split("\n")

    for ln in lines:
        prefix_x = x + (30 if bullet else 0)
        prefix = "• " if (bullet and ln.strip()) else ""
        draw.text((prefix_x, y), prefix + ln, font=font, fill=fill)
        y += font.size + spacing
    return y

def draw_box(x1, y1, x2, y2, title, bullets, fill=BOX_FILL, outline=BOX_OUTLINE,
             header_fill=TITLE_BAR_FILL, title_color=TITLE_TEXT):
    """
    Draws a rounded rectangle with a title bar and bullet list inside.
    Returns the bottom y position after the content.
    """
    # Container
    draw.rounded_rectangle([x1, y1, x2, y2], radius=22, fill=fill, outline=outline, width=4)
    # Title bar
    draw.rounded_rectangle([x1, y1, x2, y1 + 80], radius=22, fill=header_fill, outline=outline, width=4)
    draw.text((x1 + 28, y1 + 18), title, font=HEADER_FONT, fill=title_color)

    # Content
    y = y1 + 100
    max_w = (x2 - x1) - 50
    for line in bullets:
        y = multiline(line, x1 + 28, y, BODY_FONT, "black", max_w=max_w, bullet=True, spacing=6)
    return y

def arrow(frm, to, color=ARROW_COLOR, width=5):
    """
    Draws a line with an arrow head from frm (x,y) to to (x,y).
    """
    draw.line([frm, to], fill=color, width=width)
    import math
    ax, ay = to
    bx, by = frm
    ang = math.atan2(ay - by, ax - bx)
    L = 22  # arrow head length
    a = 0.55  # arrow head angle
    x1 = ax - L * math.cos(ang - a)
    y1 = ay - L * math.sin(ang - a)
    x2 = ax - L * math.cos(ang + a)
    y2 = ay - L * math.sin(ang + a)
    draw.polygon([(ax, ay), (x1, y1), (x2, y2)], fill=color)

# ----------------------------
# Title
# ----------------------------
draw.text((40, 40), TITLE, font=TITLE_FONT, fill="black")

# ----------------------------
# Layout (v6): DT_LOCATION & DT_WELL are 3x width and 2x height
# ----------------------------
# Top (widened and taller)
loc = (450, 130, 1550, 350)   # DT_LOCATION (double height)
well = (450, 380, 1550, 600)  # DT_WELL (double height)

# Middle row
coord = (80, 700, 750, 1000)    # DT_COORDINATE (left)
md = (830, 700, 1230, 1130)     # DT_MEASURED_DATUM (middle)
seg = (1300, 700, 1920, 1000)   # DT_WELL_SEGMENT (right)

# Bottom derived calculations
calc = (230, 1180, 1770, 1460)

# ----------------------------
# Draw Boxes
# ----------------------------
draw_box(*loc, "DT_LOCATION", [
    "Stores sys_loc_code (primary key across all EQuIS tables)",
    "Required parent for wells and all location-based data",
    "Defines location identity for dependent tables"
])

draw_box(*well, "DT_WELL", [
    "Well metadata (ID, type, status, installation date)",
    "Does not store coordinates, elevations, TOC, or screen data"
])

draw_box(*coord, "DT_COORDINATE", [
    "X/Y coordinates and ground surface elevation (GS)",
    "Single authoritative record; overwrite with latest survey",
    "Optional: coordinate type, datum, accuracy"
])

draw_box(*md, "DT_MEASURED_DATUM", [
    "Reference datum (usually TOC) for water-level calculations",
    "start_date allows historical & future TOC changes",
    "WL elevations use TOC active on measurement date"
])

draw_box(*seg, "DT_WELL_SEGMENT", [
    "Construction intervals as depths (ft bgs)",
    "Store raw depths only; do not store calculated elevations",
    "Screen elevations derived from GS – depth (bgs)"
])

draw_box(*calc, "Derived Calculations (Not Stored)", [
    "Water-Level Elevation = TOC Elevation – DTW",
    "Screen Top Elevation = GS Elevation – Screen Top Depth (bgs)",
    "Screen Bottom Elevation = GS Elevation – Screen Bottom Depth (bgs)"
], fill=CALC_FILL, outline=CALC_OUTLINE)

# ----------------------------
# Connectors (arrows)
# ----------------------------
# DT_LOCATION -> DT_WELL
arrow(((loc[0] + loc[2]) // 2, loc[3]), ((well[0] + well[2]) // 2, well[1]))

# DT_WELL -> middle row
arrow(((well[0] + well[2]) // 2, well[3]), ((coord[0] + coord[2]) // 2, coord[1]))
arrow(((well[0] + well[2]) // 2, well[3]), ((md[0] + md[2]) // 2, md[1]))
arrow(((well[0] + well[2]) // 2, well[3]), ((seg[0] + seg[2]) // 2, seg[1]))

# middle row -> Derived Calculations
arrow(((coord[0] + coord[2]) // 2, coord[3]), ((calc[0] + calc[2]) // 2 - 500, calc[1]))
arrow(((md[0] + md[2]) // 2, md[3]), ((calc[0] + calc[2]) // 2, calc[1]))
arrow(((seg[0] + seg[2]) // 2, seg[3]), ((calc[0] + calc[2]) // 2 + 500, calc[1]))

# ----------------------------
# Footer
# ----------------------------
draw.text((40, CANVAS_H - 40), FOOTER_TEXT, font=SMALL_FONT, fill=FOOTER_FILL)

# Save
img.save(OUTFILE)
print(f"Saved: {OUTFILE}")

Saved: EQuIS_Well_Data_Relationships_v6.png
